# AI Facial Emotion Detection System
### Final Year B.Tech Computer Science Project

---

## What does this project do?
This notebook detects **7 human emotions** from real face photographs using Artificial Intelligence.

| Emotion   | What it looks like             |
|-----------|--------------------------------|
| Angry     | Frowning, eyebrows pulled down |
| Disgust   | Wrinkled nose, curled lip      |
| Fear      | Wide eyes, raised eyebrows     |
| Happy     | Smiling, raised cheeks         |
| Neutral   | Relaxed face, no expression    |
| Sad       | Downturned mouth, drooping eyes|
| Surprise  | Wide eyes, raised brows, open mouth |

## Why is emotion detection important?
- **Mental health monitoring** — detect stress or depression early
- **Human-computer interaction** — make computers respond to your mood
- **Education** — detect if students are confused or engaged
- **Customer service** — understand customer satisfaction

## Dataset
We use **FER2013** — a standard benchmark dataset with **35,887 real grayscale human face photos** (48×48 pixels), labeled by human annotators into 7 emotions.

## How to run this notebook
Run cells **one by one from top to bottom** using `Shift+Enter`. Each cell builds on the previous one.

## Step 1 — Install Required Packages

Run this once to install everything needed. Takes about 1 minute.

In [ ]:
import subprocess, sys

packages = [
    'torch', 'torchvision',
    'transformers',      # Pre-trained ViT emotion model from HuggingFace
    'datasets',          # Load FER2013 dataset directly from HuggingFace (no Kaggle needed)
    'Pillow',            # Image handling
    'opencv-python',     # Face detection in your own photos
    'matplotlib', 'seaborn', 'scikit-learn', 'numpy',
]

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install'] + packages + ['-q'],
    capture_output=True, text=True
)
print('All packages installed successfully!')
print('You can now run the remaining cells.')

## Step 2 — Import Libraries & Set Up

In [ ]:
import os, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from datasets import load_dataset
from transformers import pipeline as hf_pipeline
from collections import Counter

warnings.filterwarnings('ignore')

# The 7 emotions we detect
EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

# Use GPU (faster) if available, otherwise CPU
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device  : {DEVICE}')
if DEVICE == 'cpu':
    print('  TIP: For faster CNN training, use Google Colab (free GPU)')
print(f'PyTorch : {torch.__version__}')
print(f'Emotions: {EMOTIONS}')

## Step 3 — Load Real FER2013 Face Images

We download the **FER2013 dataset** directly from HuggingFace — no Kaggle account needed.

- **7,178 real grayscale face photos** (48×48 pixels)
- Photos collected from Google Images and labeled by human annotators
- This is the standard test set of FER2013

> First download takes 1-2 minutes (~50 MB). Future runs use the cached version.

In [ ]:
print('Downloading FER2013 real face images from HuggingFace...')
print('(~50 MB download — only the first time)')
print()

fer_dataset = load_dataset('Piro17/fer2013test', split='train')

print(f'Loaded   : {len(fer_dataset):,} real face images')
print(f'Size     : {fer_dataset[0]["image"].size} pixels')
print(f'Mode     : {fer_dataset[0]["image"].mode}  (L = grayscale)')
print()

# Show class distribution
label_counts = Counter(fer_dataset['label'])
print('Class distribution (images per emotion):')
print(f'  {"Emotion":12s} {"Images":>7s}  Bar Chart')
print(f'  {"-"*12} {"-"*7}  {"-"*30}')
for i, name in enumerate(EMOTIONS):
    count = label_counts[i]
    bar   = '█' * int(count / 30)
    print(f'  {name.capitalize():12s} {count:7d}  {bar}')
print(f'  {"TOTAL":12s} {len(fer_dataset):7d}')

## Step 4 — Visualise Real Face Samples

See what actual FER2013 images look like. We show **4 samples per emotion**.

**Notice** how some emotions (Fear vs Surprise, Sad vs Neutral) look very similar — this is why emotion detection is a hard problem!

In [ ]:
# Collect 4 real face samples per emotion class
samples_per_class = {i: [] for i in range(7)}
for item in fer_dataset:
    lbl = item['label']
    if len(samples_per_class[lbl]) < 4:
        samples_per_class[lbl].append(item['image'])
    if all(len(v) == 4 for v in samples_per_class.values()):
        break

fig, axes = plt.subplots(4, 7, figsize=(14, 8))
fig.patch.set_facecolor('#f8f8f8')

for col, emotion in enumerate(EMOTIONS):
    for row in range(4):
        ax  = axes[row][col]
        img = samples_per_class[col][row]
        ax.imshow(img, cmap='gray', vmin=0, vmax=255)
        ax.axis('off')
        if row == 0:
            ax.set_title(emotion.capitalize(), fontsize=11,
                         fontweight='bold', pad=5)

plt.suptitle('Real FER2013 Human Face Photographs — 4 Samples per Emotion',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('real_face_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: real_face_samples.png')
print()
print('Key observations:')
print('  - Fear and Surprise look very similar (both have wide eyes)')
print('  - Disgust has the fewest samples (class imbalance)')
print('  - Happy is the easiest to recognize (distinct smile)')

## Step 5 — Pre-trained ViT Emotion Model

We use a **Vision Transformer (ViT)** model that was already trained on FER2013.

### What is a Vision Transformer?
- ViT splits the face image into small patches (like puzzle pieces)
- It learns relationships between patches to understand the whole face
- Model: `trpakov/vit-face-expression` — trained on 28,709 real FER2013 faces

### Why use a pre-trained model?
- Training from scratch on FER2013 takes hours on a GPU
- A pre-trained model already knows how faces look
- We just run inference (prediction) — no training needed!

> First load downloads ~300 MB model weights. Future runs use the cache.

In [ ]:
print('Loading pre-trained ViT emotion model from HuggingFace...')
print('(~300 MB download on first run)')
print()

emotion_pipe = hf_pipeline(
    'image-classification',
    model='trpakov/vit-face-expression',
    device=-1,   # CPU
)

print('Model loaded!')
print(f'Model labels: {emotion_pipe.model.config.id2label}')

## Step 6 — Test Pre-trained Model on Real Faces

Let's see the ViT model predict emotions on real FER2013 photos.

In [ ]:
def predict_and_show(image, actual_label=None, title=''):
    """Predict emotion for one face and display confidence chart."""
    img_rgb   = image.convert('RGB')   # ViT needs RGB (3 channels)
    results   = emotion_pipe(img_rgb)

    # Build score array in EMOTIONS order
    score_map = {r['label']: r['score'] for r in results}
    scores    = [score_map.get(e, 0.0) for e in EMOTIONS]
    predicted = max(results, key=lambda x: x['score'])['label']
    conf      = max(results, key=lambda x: x['score'])['score'] * 100

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))

    # Face image
    ax1.imshow(image, cmap='gray')
    status = ''
    if actual_label is not None:
        status = ' ✓' if predicted == actual_label else f' ✗ (actual: {actual_label})'
    ax1.set_title(f'Predicted: {predicted.upper()} ({conf:.1f}%){status}', fontsize=11)
    ax1.axis('off')

    # Confidence bars
    colors = ['tomato' if e == predicted else 'steelblue' for e in EMOTIONS]
    ax2.barh([e.capitalize() for e in EMOTIONS], [s*100 for s in scores], color=colors)
    ax2.set_xlabel('Confidence (%)')
    ax2.set_title('Confidence for All 7 Emotions')
    ax2.set_xlim(0, 110)
    for i, s in enumerate(scores):
        if s > 0.01:
            ax2.text(s*100 + 0.5, i, f'{s*100:.1f}%', va='center', fontsize=9)

    if title:
        plt.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    return predicted

# Test on one image per emotion
print('Testing pre-trained ViT model on one real face per emotion:')
print()
for idx, name in enumerate(EMOTIONS):
    for item in fer_dataset:
        if item['label'] == idx:
            predict_and_show(item['image'],
                             actual_label=name,
                             title=f'Real FER2013 — Actual: {name.upper()}')
            break

## Step 7 — Evaluate Pre-trained Model on 700 Real Faces

We test the model on **100 images per emotion** (700 total) to measure real accuracy.

> **Expected accuracy: 65-72%** — this matches the FER2013 benchmark.
> Human accuracy on FER2013 is ~65%, so this is very good!

> Takes ~3-5 minutes on CPU.

In [ ]:
N_PER_CLASS = 100   # 100 images × 7 classes = 700 total

# Collect balanced subset
eval_images  = []
eval_labels  = []
class_counts = {i: 0 for i in range(7)}

for item in fer_dataset:
    lbl = item['label']
    if class_counts[lbl] < N_PER_CLASS:
        eval_images.append(item['image'])
        eval_labels.append(lbl)
        class_counts[lbl] += 1
    if all(v == N_PER_CLASS for v in class_counts.values()):
        break

print(f'Evaluating on {len(eval_images)} real face images...')
print('This may take a few minutes on CPU.\n')

# Batch predictions for speed
vit_preds = []
BATCH = 20
for i in range(0, len(eval_images), BATCH):
    batch   = [img.convert('RGB') for img in eval_images[i:i+BATCH]]
    results = emotion_pipe(batch)
    for res in results:
        best = max(res, key=lambda x: x['score'])['label']
        vit_preds.append(EMOTIONS.index(best))
    done = min(i + BATCH, len(eval_images))
    print(f'  Progress: {done}/{len(eval_images)} images', end='\r')

vit_preds  = np.array(vit_preds)
vit_labels = np.array(eval_labels)
vit_acc    = (vit_preds == vit_labels).mean() * 100

print(f'\n\n=== Pre-trained ViT Results on Real FER2013 Faces ===')
print(f'Overall Accuracy: {vit_acc:.1f}%')
print()
print(classification_report(
    vit_labels, vit_preds,
    target_names=[e.capitalize() for e in EMOTIONS],
    digits=3
))

## Step 8 — Confusion Matrix (Pre-trained ViT)

In [ ]:
cm = confusion_matrix(vit_labels, vit_preds)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True, fmt='d', cmap='Blues',
    xticklabels=[e.capitalize() for e in EMOTIONS],
    yticklabels=[e.capitalize() for e in EMOTIONS],
    linewidths=0.5, linecolor='lightgray'
)
plt.xlabel('Predicted Emotion', fontsize=12)
plt.ylabel('Actual Emotion',    fontsize=12)
plt.title(
    f'Confusion Matrix — Pre-trained ViT Model\n'
    f'Real FER2013 Faces  (Overall Accuracy: {vit_acc:.1f}%)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('confusion_matrix_vit.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix_vit.png')

# Per-class accuracy
print('\nPer-class accuracy:')
for i, name in enumerate(EMOTIONS):
    mask = vit_labels == i
    acc  = (vit_preds[mask] == i).mean() * 100
    bar  = '█' * int(acc / 5)
    print(f'  {name.capitalize():10s}: {acc:5.1f}%  {bar}')

# Most confused pair
cm2 = cm.copy(); np.fill_diagonal(cm2, 0)
r, c = np.unravel_index(cm2.argmax(), cm2.shape)
print(f'\nMost confused: [{EMOTIONS[r].upper()}] misclassified as [{EMOTIONS[c].upper()}] ({cm2[r,c]} times)')
print('(This is typical — Fear and Surprise both involve wide eyes)')

## Step 9 — Build a CNN from Scratch

Now we **build and train our own Convolutional Neural Network (CNN)**.
This teaches you how AI learns to recognize emotions from scratch.

### How a CNN works:
```
Face Photo (48×48)  →  Block 1: Detect Edges  →  Block 2: Detect Face Parts  →  Emotion
  1 channel            32 feature maps (24×24)    64 feature maps (12×12)        7 scores
```

### Data split:
- **80% (5,742 images) for training** — the model learns from these
- **20% (1,436 images) for testing** — we measure accuracy on these
- Using **stratified split** so every emotion class is equally represented in both sets

In [ ]:
# ─── 1. Stratified 80/20 split ─────────────────────────────────────────────
# IMPORTANT: The dataset is sorted by class — we must shuffle before splitting!
# Stratified split ensures each emotion class is equally in train and test.

all_labels  = np.array(fer_dataset['label'])
all_indices = np.arange(len(fer_dataset))

train_idx, test_idx = train_test_split(
    all_indices,
    test_size=0.2,
    random_state=42,
    stratify=all_labels,     # ensures each class has same proportion in train/test
)

print(f'Dataset size : {len(fer_dataset):,} images')
print(f'Train size   : {len(train_idx):,} images (80%)')
print(f'Test  size   : {len(test_idx):,} images (20%)')
print()
print('Checking stratification (samples per class in train/test):')
print(f'  {"Emotion":12s}  {"Train":>6s}  {"Test":>5s}')
train_lbls = all_labels[train_idx]
test_lbls  = all_labels[test_idx]
for i, name in enumerate(EMOTIONS):
    n_tr = (train_lbls == i).sum()
    n_te = (test_lbls  == i).sum()
    print(f'  {name.capitalize():12s}  {n_tr:6d}  {n_te:5d}')

# ─── 2. Dataset wrapper ──────────────────────────────────────────────────────
class FERDataset(Dataset):
    def __init__(self, hf_dataset, indices, transform):
        self.data      = hf_dataset
        self.indices   = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        item  = self.data[int(self.indices[i])]
        image = item['image']      # PIL 48×48 grayscale
        label = item['label']      # 0-6
        if self.transform:
            image = self.transform(image)
        return image, label

# Training transform: augment data to prevent overfitting
train_tf = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((48, 48)),
    transforms.RandomHorizontalFlip(p=0.5),     # randomly flip face left/right
    transforms.RandomRotation(degrees=10),       # randomly rotate ±10 degrees
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]), # scale pixels to [-1, +1]
])

test_tf = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

train_ds = FERDataset(fer_dataset, train_idx, train_tf)
test_ds  = FERDataset(fer_dataset, test_idx,  test_tf)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=0)

print(f'\nDataLoaders ready!')
print(f'Training batches: {len(train_loader)}')
print(f'Testing  batches: {len(test_loader)}')

In [ ]:
# ─── CNN Architecture ──────────────────────────────────────────────────────

class EmotionCNN(nn.Module):
    """
    Simple 2-block CNN for emotion recognition.

    Input  : (batch, 1, 48, 48) — grayscale face images
    Output : (batch, 7)         — scores for each emotion

    Architecture:
        Block1: Conv(1→32)  + BN + ReLU + MaxPool(2) + Dropout(0.25)  → 24×24
        Block2: Conv(32→64) + BN + ReLU + MaxPool(2) + Dropout(0.25)  → 12×12
        Head  : Flatten → Linear(9216→256) → ReLU → Dropout(0.5) → Linear(256→7)
    """
    def __init__(self):
        super().__init__()

        # Block 1: detects low-level features (edges, gradients) → 48×48 → 24×24
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),   # normalise activations, speeds up learning
            nn.ReLU(),
            nn.MaxPool2d(2),      # halve spatial size
            nn.Dropout2d(0.25),   # drop 25% of feature maps to prevent overfitting
        )

        # Block 2: detects facial parts (eyes, nose, mouth) → 24×24 → 12×12
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),
        )

        # Classification head: combine features into 7 emotion scores
        self.head = nn.Sequential(
            nn.Flatten(),                         # 64×12×12 = 9,216 values
            nn.Linear(64 * 12 * 12, 256),
            nn.ReLU(),
            nn.Dropout(0.5),                      # strong dropout (50%) before final layer
            nn.Linear(256, 7),                    # 7 emotion classes
        )

    def forward(self, x):
        return self.head(self.block2(self.block1(x)))


cnn = EmotionCNN().to(DEVICE)
n_params = sum(p.numel() for p in cnn.parameters())
print(f'CNN model created!')
print(f'Total parameters: {n_params:,}')
print()
print(cnn)

## Step 10 — Train the CNN

Training means: show the CNN each face image and adjust its internal numbers (weights) so it correctly identifies the emotion.

**Expected accuracy after training: 60–66%** — this is realistic for FER2013.
> Note: Human accuracy on FER2013 is only ~65%. FER2013 has many ambiguous images.

> On CPU: ~2-3 min/epoch × 20 epochs = 40-60 min total
> On Google Colab GPU: ~20-30 seconds/epoch

In [ ]:
EPOCHS    = 20
PATIENCE  = 5    # stop if val accuracy doesn't improve for 5 consecutive epochs

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.5
)

history      = {'train_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
best_weights = {k: v.clone() for k, v in cnn.state_dict().items()}  # init with current weights
patience_cnt = 0

print(f'Training CNN on real FER2013 faces for up to {EPOCHS} epochs...')
print(f'Expected accuracy: 60-66%  (human accuracy on FER2013 = 65%)')
print()
print(f'  {"Epoch":>5} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Acc":>8}')
print(f'  {"-"*5}-+-{"-"*10}-+-{"-"*9}-+-{"-"*8}')

for epoch in range(1, EPOCHS + 1):

    # ── Train phase ──────────────────────────────────────────────────────
    cnn.train()
    total_loss, correct = 0.0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(cnn(imgs), lbls)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += (cnn(imgs).argmax(1) == lbls).sum().item()
    train_acc  = correct / len(train_ds) * 100
    train_loss = total_loss / len(train_loader)

    # ── Validation phase ──────────────────────────────────────────────────
    cnn.eval()
    val_correct = 0
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            val_correct += (cnn(imgs).argmax(1) == lbls).sum().item()
    val_acc = val_correct / len(test_ds) * 100

    scheduler.step(val_acc)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    # ── Save best model ──────────────────────────────────────────────────
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_weights = {k: v.clone() for k, v in cnn.state_dict().items()}
        patience_cnt = 0
        tag = ' ← best'
    else:
        patience_cnt += 1
        tag = ''

    print(f'  {epoch:>5} | {train_loss:>10.4f} | {train_acc:>8.2f}% | {val_acc:>7.2f}%{tag}')

    if patience_cnt >= PATIENCE:
        print(f'\n  Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)')
        break

# Restore best weights
cnn.load_state_dict(best_weights)
print(f'\nTraining complete!')
print(f'Best validation accuracy: {best_val_acc:.2f}%')
torch.save(cnn.state_dict(), 'emotion_cnn_model.pth')
print('Model saved: emotion_cnn_model.pth')

In [ ]:
# ─── Plot Training Curves ───────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

epochs_done = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs_done, history['train_loss'], color='steelblue', linewidth=2)
ax1.set_title('Training Loss over Epochs', fontsize=12)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-Entropy Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_done, history['train_acc'], label='Train Accuracy', color='steelblue', linewidth=2)
ax2.plot(epochs_done, history['val_acc'],   label='Val Accuracy',   color='tomato',    linewidth=2)
ax2.axhline(65, color='green', linestyle='--', linewidth=1, label='Human accuracy (65%)')
ax2.set_title('Accuracy over Epochs', fontsize=12)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('CNN Training on Real FER2013 Human Face Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cnn_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: cnn_training_curves.png')

## Step 11 — Evaluate CNN on Real Faces

In [ ]:
cnn.eval()
cnn_preds, cnn_true = [], []

with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs = imgs.to(DEVICE)
        cnn_preds.extend(cnn(imgs).argmax(1).cpu().numpy())
        cnn_true.extend(lbls.numpy())

cnn_preds = np.array(cnn_preds)
cnn_true  = np.array(cnn_true)
cnn_acc   = (cnn_preds == cnn_true).mean() * 100

print(f'=== CNN Results on Real FER2013 Faces ===')
print(f'Test Accuracy: {cnn_acc:.2f}%')
print()
print(classification_report(
    cnn_true, cnn_preds,
    target_names=[e.capitalize() for e in EMOTIONS],
    digits=3
))

# Confusion matrix
cm_cnn = confusion_matrix(cnn_true, cnn_preds)
plt.figure(figsize=(9, 7))
sns.heatmap(
    cm_cnn, annot=True, fmt='d', cmap='Greens',
    xticklabels=[e.capitalize() for e in EMOTIONS],
    yticklabels=[e.capitalize() for e in EMOTIONS],
    linewidths=0.5
)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual',    fontsize=12)
plt.title(
    f'CNN Confusion Matrix — Real FER2013 Faces\n'
    f'(Test Accuracy: {cnn_acc:.1f}%)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('confusion_matrix_cnn.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix_cnn.png')

## Step 12 — Test on YOUR OWN Photo

1. Copy a face photo to the same folder as this notebook
2. Change `MY_PHOTO` below to your photo filename
3. Run the cell

The system will **automatically detect your face** using OpenCV Haar Cascade, then predict your emotion.

In [ ]:
import cv2

# ─── Change this to your photo file path ────────────────────────────────────
MY_PHOTO = 'my_photo.jpg'
# ────────────────────────────────────────────────────────────────────────────

def predict_my_photo(image_path):
    """Detect face(s) in photo and predict emotion for each."""
    # Load image
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        print(f'Cannot read: {image_path}')
        print('Make sure the file exists in the same folder as this notebook.')
        return

    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # OpenCV Haar Cascade face detector
    detector = cv2.CascadeClassifier(
        cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
    )
    faces = detector.detectMultiScale(
        img_gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30)
    )

    if len(faces) == 0:
        print('No face detected. Tips:')
        print('  - Use a clear, forward-facing photo')
        print('  - Ensure good lighting')
        print('  - Face should be a significant part of the image')
        return

    print(f'Detected {len(faces)} face(s)!')

    for i, (x, y, w, h) in enumerate(faces):
        face_crop = img_gray[y:y+h, x:x+w]
        face_pil  = Image.fromarray(face_crop).resize((48, 48))

        print(f'\nFace {i+1}:')
        # Use pre-trained ViT (more accurate than our small CNN)
        predict_and_show(face_pil, title=f'Your Photo — Face {i+1}')


if os.path.exists(MY_PHOTO):
    predict_my_photo(MY_PHOTO)
else:
    print(f'Photo not found: {MY_PHOTO}')
    print()
    print('HOW TO TEST YOUR OWN PHOTO:')
    print('  1. Copy any face photo to this folder')
    print('  2. Change MY_PHOTO above to match your filename')
    print('  3. Run this cell again')
    print()
    print('Showing a demo on a real FER2013 image instead:')
    demo = fer_dataset[100]
    predict_and_show(demo['image'],
                     actual_label=EMOTIONS[demo['label']],
                     title='Demo — Real FER2013 Face')

## Step 13 — Final Results Summary

In [ ]:
print('=' * 60)
print('           AI EMOTION DETECTION — FINAL SUMMARY')
print('=' * 60)
print()
print('DATASET')
print(f'  Name    : FER2013 (real human face photographs)')
print(f'  Images  : {len(fer_dataset):,} total (48×48 grayscale)')
print(f'  Classes : 7 emotions (Angry, Disgust, Fear, Happy,')
print(f'            Neutral, Sad, Surprise)')
print(f'  Source  : HuggingFace — Piro17/fer2013test')
print()
print('TRAIN/TEST SPLIT')
print(f'  Train   : {len(train_idx):,} images (80% — stratified)')
print(f'  Test    : {len(test_idx):,} images (20% — stratified)')
print()
print('MODEL 1 — Pre-trained Vision Transformer (ViT)')
print(f'  Model   : trpakov/vit-face-expression')
print(f'  Trained : FER2013 full training set (28,709 real faces)')
print(f'  Accuracy: {vit_acc:.1f}% on 700 real FER2013 images')
print()
print('MODEL 2 — Custom CNN (trained from scratch)')
print(f'  Architecture: 2 Conv blocks + Dropout + FC layers')
print(f'  Parameters  : {n_params:,}')
print(f'  Epochs      : {len(history["train_acc"])} (with early stopping)')
print(f'  Accuracy    : {cnn_acc:.1f}% on {len(test_idx):,} real FER2013 images')
print()
print('CONTEXT')
print(f'  Human accuracy on FER2013    : ~65%')
print(f'  State-of-the-art (2024)      : ~76%')
print(f'  Our ViT model                : {vit_acc:.1f}%')
print(f'  Our CNN (scratch, 7K images) : {cnn_acc:.1f}%')
print()
print('OUTPUT FILES SAVED')
for fname in ['real_face_samples.png', 'confusion_matrix_vit.png',
              'confusion_matrix_cnn.png', 'cnn_training_curves.png',
              'emotion_cnn_model.pth']:
    size = os.path.getsize(fname) // 1024 if os.path.exists(fname) else 0
    marker = 'KB' if os.path.exists(fname) else '(not yet saved)'
    print(f'  {fname:<35} {size} {marker}')